# Governed Foundry Hosted Agent with AGT + Citadel

**Agent Governance Toolkit x Citadel Integration**

This notebook demonstrates AGT governance for
[Foundry hosted agents](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/hosted-agents/agent-framework/responses).

**Flow:**
1. Generate the governed agent code (Foundry agent + AGT wrapper)
2. Run AGT governance locally to demonstrate allow/block decisions
3. (Optional) Deploy to Foundry with `azd` and call the live agent
4. (Optional) Export governance events to Citadel

**Prerequisites:**
- `pip install agent-os-kernel` (for local governance simulation)
- Azure AI Foundry project + `azd` (for deployment only)
- (Optional) Citadel deployment with Event Hub for governance telemetry

```
This Notebook (client)           Foundry (server)
   |                                |
   |--- POST /responses ----------->|
   |                         [AGT Kernel checks policy]
   |                           /           \
   |                      ALLOW             DENY
   |                        |                 |
   |                   Execute tool     Block + audit
   |                        |                 |
   |<-- response -----------+-----------------|
   |                        |
   |               [Citadel Exporter]
   |                        |
   |               Event Hub / App Insights
```

## Step 1: Install Dependencies

AGT kernel runs locally for governance simulation.
No Foundry SDK needed until deployment.

In [ ]:
!pip install agent-os-kernel requests -q

## Step 2: Configure Environment

In [ ]:
import os

# Required: Foundry project settings (for deployment only)
FOUNDRY_PROJECT_ENDPOINT = os.environ.get(
    "FOUNDRY_PROJECT_ENDPOINT",
    "https://<your-project>.services.ai.azure.com/api"
)
MODEL_DEPLOYMENT = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

# After deployment, set to your hosted agent endpoint
AGENT_ENDPOINT = os.environ.get(
    "AGENT_ENDPOINT",
    "http://localhost:8088"  # Local dev; replaced after azd up
)

print(f"Foundry endpoint:  {FOUNDRY_PROJECT_ENDPOINT}")
print(f"Model deployment:  {MODEL_DEPLOYMENT}")
print(f"Agent endpoint:    {AGENT_ENDPOINT}")

## Step 3: Generate the Governed Agent Code

This cell writes the agent source files to disk. The agent is a standard
Foundry hosted agent (Responses protocol) with AGT governance wrapping
every tool call.

AGT runs **inside** the container, so governance is enforced server-side
before any tool executes.

In [ ]:
import os, textwrap, pathlib

AGENT_DIR = pathlib.Path("citadel-governed-agent")
AGENT_DIR.mkdir(exist_ok=True)

# --- main.py: Foundry hosted agent with AGT governance ---
(AGENT_DIR / "main.py").write_text(textwrap.dedent('''\
    # Copyright (c) Microsoft Corporation. Licensed under the MIT License.
    """Foundry hosted agent with AGT governance."""
    import os
    from random import randint
    from typing import Annotated

    from agent_framework import Agent, tool
    from agent_framework.foundry import FoundryChatClient
    from agent_framework_foundry_hosting import ResponsesHostServer
    from azure.identity import DefaultAzureCredential
    from pydantic import Field

    from agent_os.stateless import StatelessKernel, ExecutionContext


    # ---- Governance kernel ----
    kernel = StatelessKernel()
    ctx = ExecutionContext(
        agent_id="citadel-foundry-agent",
        policies=["read_only", "no_pii"],
    )


    # ---- Tools (governed) ----
    @tool(approval_mode="never_require")
    async def get_weather(
        location: Annotated[str, Field(description="Location for weather.")],
    ) -> str:
        """Get the weather for a given location."""
        result = await kernel.execute("get_weather", {"location": location}, ctx)
        if not result.success:
            return f"[AGT BLOCKED] {result.error}"
        conditions = ["sunny", "cloudy", "rainy", "stormy"]
        return f"Weather in {location}: {conditions[randint(0, 3)]}, high {randint(10, 30)}C."


    @tool(approval_mode="never_require")
    async def send_email(
        to: Annotated[str, Field(description="Recipient email.")],
        body: Annotated[str, Field(description="Email body.")],
    ) -> str:
        """Send an email."""
        result = await kernel.execute("send_email", {"to": to, "body": body}, ctx)
        if not result.success:
            return f"[AGT BLOCKED] {result.error}"
        return f"Email sent to {to}"


    @tool(approval_mode="never_require")
    async def lookup_customer(
        customer_id: Annotated[str, Field(description="Customer ID.")],
    ) -> str:
        """Look up customer information."""
        result = await kernel.execute("lookup_customer", {"customer_id": customer_id}, ctx)
        if not result.success:
            return f"[AGT BLOCKED] {result.error}"
        return f"Customer {customer_id}: Contoso Corp, Active"


    def main():
        client = FoundryChatClient(
            project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
            credential=DefaultAzureCredential(),
        )
        agent = Agent(
            client=client,
            instructions=(
                "You are a customer service assistant. "
                "You can check the weather and look up customers. "
                "Keep answers brief. If a tool is blocked, explain why."
            ),
            tools=[get_weather, send_email, lookup_customer],
            default_options={"store": False},
        )
        server = ResponsesHostServer(agent)
        server.run()


    if __name__ == "__main__":
        main()
'''))

# --- requirements.txt ---
(AGENT_DIR / "requirements.txt").write_text(textwrap.dedent('''\
    agent-framework>=1.2.2
    agent-framework-foundry-hosting
    agent-os-kernel
'''))

# --- Dockerfile ---
(AGENT_DIR / "Dockerfile").write_text(textwrap.dedent('''\
    FROM python:3.12-slim
    WORKDIR /app
    COPY . user_agent/
    WORKDIR /app/user_agent
    RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
    EXPOSE 8088
    CMD ["python", "main.py"]
'''))

# --- agent.yaml ---
(AGENT_DIR / "agent.yaml").write_text(textwrap.dedent('''\
    kind: hosted
    name: citadel-governed-agent
    protocols:
      - protocol: responses
        version: 1.0.0
    resources:
      cpu: "0.25"
      memory: "0.5Gi"
    environment_variables:
      - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
        value: ${AZURE_AI_MODEL_DEPLOYMENT_NAME}
'''))

# --- agent.manifest.yaml ---
(AGENT_DIR / "agent.manifest.yaml").write_text(textwrap.dedent('''\
    name: citadel-governed-agent
    description: >
      Foundry hosted agent with AGT governance. Demonstrates policy enforcement,
      tool gating, PII blocking, and Citadel telemetry export.
    metadata:
      tags:
        - Agent Framework
        - Agent Governance Toolkit
        - Citadel
    template:
      name: citadel-governed-agent
      kind: hosted
      protocols:
        - protocol: responses
          version: 1.0.0
      environment_variables:
        - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
          value: "{{AZURE_AI_MODEL_DEPLOYMENT_NAME}}"
    resources:
      - kind: model
        id: gpt-4o
        name: AZURE_AI_MODEL_DEPLOYMENT_NAME
'''))

print("Agent files generated:")
for f in sorted(AGENT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size} bytes)")

## Step 4: Deploy to Foundry (Optional)

Deploy the governed agent to Azure AI Foundry using `azd`.
After deployment, update `AGENT_ENDPOINT` with the hosted URL.

> **You can skip deployment** and proceed to Step 5 to see governance
> decisions locally. The kernel runs the same logic as the hosted agent.

In [ ]:
# Option A: Deploy with azd (recommended)
# Run these commands in your terminal:
#
#   cd citadel-governed-agent
#   azd ai agent init -m agent.manifest.yaml
#   azd up
#
# After deployment, copy the agent endpoint URL and set it below.

# Option B: Run locally for testing
# In a separate terminal:
#   cd citadel-governed-agent
#   pip install -r requirements.txt
#   python main.py
# The agent will be available at http://localhost:8088

# Set the endpoint after deployment
# AGENT_ENDPOINT = "https://<your-deployed-agent>.azurecontainerapps.io"

print(f"Agent endpoint: {AGENT_ENDPOINT}")
print("\nIf running locally: python citadel-governed-agent/main.py")
print("If deploying to Foundry: azd ai agent init + azd up")

## Step 5: Local Governance Simulation

Run the AGT governance kernel **locally** to demonstrate allow/block
decisions. This is the exact same logic that runs server-side inside
the Foundry container.

No deployment needed: the kernel uses an in-memory backend.

In [ ]:
from agent_os.stateless import StatelessKernel, ExecutionContext

# The same kernel + context used inside the hosted agent container
kernel = StatelessKernel()
ctx = ExecutionContext(
    agent_id="citadel-foundry-agent",
    policies=["read_only", "no_pii"],
)

print("StatelessKernel initialized (in-memory backend)")
print(f"Policies: {ctx.policies}")
print("Built-in policy rules:")
for name in ctx.policies:
    rules = kernel.policies.get(name, {})
    print(f"  {name}: {rules}")

In [ ]:
async def test_governance(action: str, params: dict, context: ExecutionContext):
    """Run a tool call through AGT governance and display the result."""
    result = await kernel.execute(action, params, context)
    status = "ALLOW" if result.success else "DENY"
    icon = "\u2705" if result.success else "\u274c"
    print(f"  {icon} {status}  {action}({params})")
    if not result.success:
        print(f"         Signal: {result.signal}")
        print(f"         Reason: {result.error}")
    return result.updated_context or context


print("=" * 60)
print("AGT GOVERNANCE SIMULATION")
print("Policy: read_only + no_pii")
print("=" * 60)

# Test 1: Allowed - get_weather is not blocked by read_only
print("\n[Test 1] Weather lookup (allowed):")
ctx = await test_governance("get_weather", {"location": "Seattle"}, ctx)

# Test 2: Blocked - send_email is in read_only blocked_actions
print("\n[Test 2] Send email (blocked by read_only policy):")
ctx = await test_governance("send_email", {"to": "user@contoso.com", "body": "Hello"}, ctx)

# Test 3: Blocked - file_write is in read_only blocked_actions
print("\n[Test 3] File write (blocked by read_only policy):")
ctx = await test_governance("file_write", {"path": "/etc/config", "content": "data"}, ctx)

# Test 4: Blocked - database_write is in read_only blocked_actions
print("\n[Test 4] Database write (blocked by read_only policy):")
ctx = await test_governance("database_write", {"table": "users", "data": "DROP TABLE"}, ctx)

# Test 5: Allowed - database read is permitted
print("\n[Test 5] Database read (allowed):")
ctx = await test_governance("database_query", {"query": "SELECT name FROM users"}, ctx)

# Test 6: Blocked - PII pattern in params
print("\n[Test 6] Response with PII (blocked by no_pii policy):")
ctx = await test_governance("respond", {"message": "Your password is abc123"}, ctx)

# Test 7: Allowed - lookup_customer is fine under read_only
print("\n[Test 7] Customer lookup (allowed):")
ctx = await test_governance("lookup_customer", {"customer_id": "C-12345"}, ctx)

## Step 6: Call the Live Hosted Agent (after deployment)

After deploying with `azd up` (Step 4), update `AGENT_ENDPOINT`
and run these cells to call the real hosted agent.

> **Skip this section** if you have not deployed the agent yet.
> Step 5 above demonstrates the same governance logic locally.

In [ ]:
import json
import requests

def call_agent(message: str, previous_response_id: str = None) -> dict:
    """Call the hosted agent via the Responses protocol."""
    payload = {"input": message}
    if previous_response_id:
        payload["previous_response_id"] = previous_response_id

    try:
        resp = requests.post(
            f"{AGENT_ENDPOINT}/responses",
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except requests.ConnectionError:
        return {
            "error": (
                f"Cannot connect to {AGENT_ENDPOINT}. "
                "Deploy the agent first (Step 4) or run locally. "
                "Step 5 demonstrates governance without deployment."
            )
        }


# Test: Allowed action - weather lookup
print("Calling hosted agent at:", AGENT_ENDPOINT)
result = call_agent("What's the weather in Seattle?")
print(json.dumps(result, indent=2))

if "error" not in result:
    # Test: Blocked action - send_email
    result = call_agent("Send an email to user@contoso.com saying hello")
    print("\nBlocked action:")
    print(json.dumps(result, indent=2))

## Step 7: Export Governance Events to Citadel (Optional)

The hosted agent can export governance events to Citadel's Event Hub.
This connects AGT (Layer 2: Agent Control Plane) to Citadel
(Layer 1: Governance Hub) through the shared observability pipeline.

To enable, add these env vars to `agent.yaml`:
```yaml
environment_variables:
  - name: CITADEL_EVENTHUB_CONN_STR
    value: "Endpoint=sb://..."
  - name: CITADEL_EVENTHUB_NAME
    value: "governance-events"
```

The cell below shows what the Citadel export produces.

In [ ]:
from agent_os.exporters.citadel_exporter import (
    GovernanceEvent,
    GovernanceEventType,
    Decision,
    CorrelationContext,
)

# Simulate the governance events the hosted agent would produce
sample_events = [
    GovernanceEvent(
        event_type=GovernanceEventType.POLICY_DECISION,
        agent_id="citadel-foundry-agent",
        action="get_weather",
        decision=Decision.ALLOW,
        policy_name="citadel-lab-policy",
        trust_score=750,
        detail="Tool in allowed_tools list",
    ),
    GovernanceEvent(
        event_type=GovernanceEventType.POLICY_VIOLATION,
        agent_id="citadel-foundry-agent",
        action="send_email",
        decision=Decision.DENY,
        policy_name="citadel-lab-policy",
        trust_score=750,
        detail="Tool not in allowed_tools: send_email",
    ),
    GovernanceEvent(
        event_type=GovernanceEventType.POLICY_VIOLATION,
        agent_id="citadel-foundry-agent",
        action="lookup_customer",
        decision=Decision.DENY,
        policy_name="citadel-lab-policy",
        trust_score=750,
        detail="Tool not in allowed_tools: lookup_customer",
    ),
]

print("Governance events (exported to Citadel Event Hub):")
print(f"{'#':<4} {'Action':<20} {'Decision':<10} {'Detail'}")
print("-" * 65)
for i, e in enumerate(sample_events, 1):
    print(f"{i:<4} {e.action:<20} {e.decision.value:<10} {e.detail}")

## Summary

| Layer | Component | Where it runs |
|-------|-----------|---------------|
| **This notebook** | Client / Governance sim | Your machine |
| **Foundry** | Hosted Agent + Tools | Azure (container) |
| **AGT** (Layer 2) | StatelessKernel + policies | Inside the container |
| **Citadel** (Layer 1) | Event Hub + Governance Hub | Azure |

**What was demonstrated:**
- `get_weather` and `lookup_customer` were **allowed** (not in read_only blocked list)
- `send_email`, `file_write`, `database_write` were **blocked** by read_only policy
- PII content was **blocked** by no_pii policy
- All decisions are logged with tamper-evident audit trails
- Governance events can be exported to Citadel via Event Hub

**Next steps:**
- Deploy to Foundry and connect to a live Citadel instance
- Customize policies (add your own rules to `StatelessKernel`)
- Enable trust scoring for continuous agent monitoring
- See the [AGT + Citadel Integration Guide](https://github.com/Azure-Samples/ai-hub-gateway-solution-accelerator/blob/citadel-v1/guides/agent-governance-toolkit-integration.md)